In [94]:
import pandas as pd
import numpy as np

In [95]:
sales = pd.read_csv("../data/raw/sales_daily.csv")
sku = pd.read_csv("../data/raw/sku_master.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")
inventory = pd.read_csv("../data/raw/inventory_snapshots.csv")

## Sales Table

In [96]:
sales

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0
...,...,...,...,...,...,...
36545,2025-12-31,SKU046,5,35877.70,7175.54,0
36546,2025-12-31,SKU047,6,47629.32,7938.22,0
36547,2025-12-31,SKU048,7,9656.85,1379.55,0
36548,2025-12-31,SKU049,16,83573.92,5223.37,0


In [97]:
sales.shape

(36550, 6)

In [98]:
sales.columns

Index(['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion'], dtype='object')

In [99]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36550 entries, 0 to 36549
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Date        36550 non-null  object 
 1   SKU         36550 non-null  object 
 2   Units_Sold  36550 non-null  int64  
 3   Revenue     36550 non-null  float64
 4   Price       36550 non-null  float64
 5   Promotion   36550 non-null  int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 1.7+ MB


In [100]:
sales.isnull().sum()

Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64

In [101]:
sales.duplicated(subset=['Date', 'SKU']).sum()

0

In [102]:
sales["Date"] = pd.to_datetime(sales["Date"], errors = "coerce") 

In [103]:
sales["Date"].dtype

dtype('<M8[ns]')

In [104]:
numeric_columns = ["Units_Sold",
                  "Revenue",
                  "Price",
                  "Promotion"]

for col in numeric_columns:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

In [105]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36550 entries, 0 to 36549
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        36550 non-null  datetime64[ns]
 1   SKU         36550 non-null  object        
 2   Units_Sold  36550 non-null  int64         
 3   Revenue     36550 non-null  float64       
 4   Price       36550 non-null  float64       
 5   Promotion   36550 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(1)
memory usage: 1.7+ MB


In [106]:
sales[sales["Units_Sold"] < 0]

,Date,SKU,Units_Sold,Revenue,Price,Promotion


In [107]:
sales[sales["Revenue"] < 0]

,Date,SKU,Units_Sold,Revenue,Price,Promotion


In [108]:
sales[sales["Price"]<= 0]

,Date,SKU,Units_Sold,Revenue,Price,Promotion


In [109]:
sales["Promotion"].value_counts(dropna=False)

Promotion
0    32800
1     3750
Name: count, dtype: int64

In [110]:
sales["Expected_Revenue"] = sales["Units_Sold"] * sales["Price"]

In [111]:
sales["Revenue_Difference"] = (sales["Revenue"] - sales["Expected_Revenue"])

In [112]:
sales["Revenue_Difference"].abs().max()

5.820766091346741e-11

In [113]:
sales.drop(columns=["Expected_Revenue", "Revenue_Difference"],
          inplace = True)

In [114]:
sales

,Date,SKU,Units_Sold,Revenue,Price,Promotion
0,2024-01-01,SKU001,5,18320.25,3664.05,0
1,2024-01-01,SKU002,15,57085.35,3805.69,0
2,2024-01-01,SKU003,5,40391.15,8078.23,0
3,2024-01-01,SKU004,5,34307.85,6861.57,0
4,2024-01-01,SKU005,12,113918.64,9493.22,0
...,...,...,...,...,...,...
36545,2025-12-31,SKU046,5,35877.70,7175.54,0
36546,2025-12-31,SKU047,6,47629.32,7938.22,0
36547,2025-12-31,SKU048,7,9656.85,1379.55,0
36548,2025-12-31,SKU049,16,83573.92,5223.37,0


In [115]:
print("Shape:", sales.shape)
print("\nMissing values:")
print(sales.isnull().sum())

print("\nDuplicate rows:")
print(sales.duplicated().sum())

print("\nDuplicate Date + SKU:")
print(sales.duplicated(subset=["Date", "SKU"]).sum())

print("\nData types:")
print(sales.dtypes)

Shape: (36550, 6)

Missing values:
Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64

Duplicate rows:
0

Duplicate Date + SKU:
0

Data types:
Date          datetime64[ns]
SKU                   object
Units_Sold             int64
Revenue              float64
Price                float64
Promotion              int64
dtype: object


## Sku_Master

In [116]:
sku.head()

,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit
0,SKU001,Product 001,Furniture,Chair,2022-04-09,1758.45,3664.05,1905.60
1,SKU002,Product 002,Home Decor,Table,2024-05-01,3867.09,3805.69,-61.40
2,SKU003,Product 003,Kitchen,Cushion,2023-12-22,589.48,8078.23,7488.75
3,SKU004,Product 004,Lighting,Cookware,2023-04-28,1445.74,6861.57,5415.83
4,SKU005,Product 005,Storage,Lamp,2023-04-22,5543.63,9493.22,3949.59


In [117]:
sku.shape

(50, 8)

In [118]:
sku.columns

Index(['SKU', 'Product_Name', 'Category', 'Subcategory', 'Launch_Date',
       'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit'],
      dtype='object')

In [119]:
sku.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SKU                    50 non-null     object 
 1   Product_Name           50 non-null     object 
 2   Category               50 non-null     object 
 3   Subcategory            50 non-null     object 
 4   Launch_Date            50 non-null     object 
 5   Cost_Price             50 non-null     float64
 6   Selling_Price          50 non-null     float64
 7   Gross_Margin_Per_Unit  50 non-null     float64
dtypes: float64(3), object(5)
memory usage: 3.3+ KB


In [120]:
sku.isnull().sum()

SKU                      0
Product_Name             0
Category                 0
Subcategory              0
Launch_Date              0
Cost_Price               0
Selling_Price            0
Gross_Margin_Per_Unit    0
dtype: int64

In [121]:
sku.duplicated().sum()

0

In [122]:
sku["SKU"].nunique()

50

In [123]:
sku["Launch_Date"] = pd.to_datetime(sku["Launch_Date"], errors = "coerce")

In [124]:
sku["Launch_Date"].dtype

dtype('<M8[ns]')

In [125]:
sku[sku["Cost_Price"] < 0]

,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit


In [126]:
sku[sku["Selling_Price"] < 0]

,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit


In [127]:
sku["Expected_Margin"] = (sku["Selling_Price"] - sku["Cost_Price"])

In [128]:
sku["Margin_Difference"] = (sku["Gross_Margin_Per_Unit"] - sku["Expected_Margin"])

In [129]:
sku["Margin_Difference"].abs().max()

9.094947017729282e-13

In [130]:
sku[sku["Gross_Margin_Per_Unit"] < 0].count()

SKU                      16
Product_Name             16
Category                 16
Subcategory              16
Launch_Date              16
Cost_Price               16
Selling_Price            16
Gross_Margin_Per_Unit    16
Expected_Margin          16
Margin_Difference        16
dtype: int64

In [131]:
sku["Category"].value_counts()

Category
Furniture     10
Home Decor    10
Kitchen       10
Lighting      10
Storage       10
Name: count, dtype: int64

In [132]:
sku["Subcategory"].value_counts()

Subcategory
Chair        5
Table        5
Cushion      5
Cookware     5
Lamp         5
Shelf        5
Cabinet      5
Rug          5
Organizer    5
Sofa         5
Name: count, dtype: int64

In [133]:
sku["SKU"] = sku["SKU"].str.strip()
sku["Product_Name"] = sku["Product_Name"].str.strip()
sku["Category"] = sku["Category"].str.strip()
sku["Subcategory"] = sku["Subcategory"].str.strip()

In [134]:
sku.drop(columns = ["Expected_Margin" , "Margin_Difference"], inplace= True)

In [135]:
print("Shape:", sku.shape)

print("\nMissing Values:")
print(sku.isnull().sum())

print("\nDuplicate rows:")
print(sku.duplicated().sum())

print("\nDuplicate SKUs:")
print(sku["SKU"].duplicated().sum())

print("\nNegative margins:")
print((sku["Gross_Margin_Per_Unit"] < 0).sum())

print("\nData types:")
print(sku.dtypes)

Shape: (50, 8)

Missing Values:
SKU                      0
Product_Name             0
Category                 0
Subcategory              0
Launch_Date              0
Cost_Price               0
Selling_Price            0
Gross_Margin_Per_Unit    0
dtype: int64

Duplicate rows:
0

Duplicate SKUs:
0

Negative margins:
16

Data types:
SKU                              object
Product_Name                     object
Category                         object
Subcategory                      object
Launch_Date              datetime64[ns]
Cost_Price                      float64
Selling_Price                   float64
Gross_Margin_Per_Unit           float64
dtype: object


In [136]:
sku["Negative_Margin"] = sku["Gross_Margin_Per_Unit"] < 0

In [137]:
sku[sku["Negative_Margin"]].head()

,SKU,Product_Name,Category,Subcategory,Launch_Date,Cost_Price,Selling_Price,Gross_Margin_Per_Unit,Negative_Margin
1,SKU002,Product 002,Home Decor,Table,2024-05-01,3867.09,3805.69,-61.40,True
6,SKU007,Product 007,Home Decor,Cabinet,2022-04-05,7748.20,5114.09,-2634.11,True
8,SKU009,Product 009,Lighting,Organizer,2022-08-10,3121.06,2336.89,-784.17,True
9,SKU010,Product 010,Storage,Sofa,2022-04-14,3889.06,663.46,-3225.60,True
10,SKU011,Product 011,Furniture,Chair,2023-08-03,1718.40,1444.56,-273.84,True


## Calendar

In [138]:
calendar.head()

,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
1,2024-01-02,2024,1,Q1,1,Tuesday,0,Winter,NaN,0,NaN
2,2024-01-03,2024,1,Q1,1,Wednesday,0,Winter,NaN,0,NaN
3,2024-01-04,2024,1,Q1,1,Thursday,0,Winter,NaN,0,NaN
4,2024-01-05,2024,1,Q1,1,Friday,0,Winter,NaN,0,NaN


In [139]:
calendar.shape

(731, 11)

In [140]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   date             731 non-null    object
 1   year             731 non-null    int64 
 2   month            731 non-null    int64 
 3   quarter          731 non-null    object
 4   week             731 non-null    int64 
 5   day_of_week      731 non-null    object
 6   is_weekend       731 non-null    int64 
 7   season           731 non-null    object
 8   holiday          8 non-null      object
 9   is_holiday       731 non-null    int64 
 10  promotion_event  75 non-null     object
dtypes: int64(5), object(6)
memory usage: 62.9+ KB


In [141]:
calendar.describe(include = "object")

,date,quarter,day_of_week,season,holiday,promotion_event
count,731,731,731,731,8,75
unique,731,4,7,5,4,1
top,2024-01-01,Q3,Monday,Monsoon,Republic Day,Seasonal Promotion
freq,1,184,105,184,2,75


In [142]:
calendar.columns

Index(['date', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend',
       'season', 'holiday', 'is_holiday', 'promotion_event'],
      dtype='object')

In [143]:
calendar.isnull().sum()

date                 0
year                 0
month                0
quarter              0
week                 0
day_of_week          0
is_weekend           0
season               0
holiday            723
is_holiday           0
promotion_event    656
dtype: int64

In [144]:
calendar.isnull().mean() * 100

date                0.000000
year                0.000000
month               0.000000
quarter             0.000000
week                0.000000
day_of_week         0.000000
is_weekend          0.000000
season              0.000000
holiday            98.905609
is_holiday          0.000000
promotion_event    89.740082
dtype: float64

In [145]:
calendar["holiday"] = calendar["holiday"].fillna("No Holiday")

In [146]:
calendar["promotion_event"] = calendar["promotion_event"].fillna("No Promotion")

In [147]:
calendar.isna().sum()

date               0
year               0
month              0
quarter            0
week               0
day_of_week        0
is_weekend         0
season             0
holiday            0
is_holiday         0
promotion_event    0
dtype: int64

In [148]:
print(calendar["promotion_event"].unique())
print(calendar["holiday"].unique())

['No Promotion' 'Seasonal Promotion']
['No Holiday' 'Republic Day' 'Independence Day' 'Diwali' 'Christmas']


In [149]:
calendar.duplicated().sum()

0

In [150]:
calendar["date"].duplicated().sum()

0

In [151]:
calendar["date"] = pd.to_datetime(
    calendar["date"],
    errors="coerce"
)

In [152]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             731 non-null    datetime64[ns]
 1   year             731 non-null    int64         
 2   month            731 non-null    int64         
 3   quarter          731 non-null    object        
 4   week             731 non-null    int64         
 5   day_of_week      731 non-null    object        
 6   is_weekend       731 non-null    int64         
 7   season           731 non-null    object        
 8   holiday          731 non-null    object        
 9   is_holiday       731 non-null    int64         
 10  promotion_event  731 non-null    object        
dtypes: datetime64[ns](1), int64(5), object(5)
memory usage: 62.9+ KB


In [153]:
calendar["date"].min(), calendar["date"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [154]:
calendar["date"].nunique()

731

In [155]:
expected_dates = pd.date_range(start = calendar["date"].min(),
                               end = calendar["date"].max(),
                               freq = "D")
expected_dates

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10',
               ...
               '2025-12-22', '2025-12-23', '2025-12-24', '2025-12-25',
               '2025-12-26', '2025-12-27', '2025-12-28', '2025-12-29',
               '2025-12-30', '2025-12-31'],
              dtype='datetime64[ns]', length=731, freq='D')

In [156]:
print("Expected dates:", len(expected_dates))
print("Unique dates:", calendar["date"].nunique())

Expected dates: 731
Unique dates: 731


In [157]:
print((calendar["year"] == calendar["date"].dt.year).all())
print((calendar["month"] == calendar["date"].dt.month).all())
print(calendar["quarter"].unique())
actual_quarter = "Q" + calendar["date"].dt.quarter.astype(str)
print((calendar["quarter"] == actual_quarter).all())
calendar["day_of_week"].unique()

True
True
['Q1' 'Q2' 'Q3' 'Q4']
True


array(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday',
       'Sunday'], dtype=object)

In [158]:
print(calendar["season"].value_counts())
print(calendar["year"].value_counts())

season
Monsoon    184
Winter     181
Spring     122
Summer     122
Autumn     122
Name: count, dtype: int64
year
2024    366
2025    365
Name: count, dtype: int64


In [159]:
calendar[calendar["is_weekend"] != (calendar["date"].dt.dayofweek >= 5)]

,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event


In [160]:
calendar.isnull().sum()

date               0
year               0
month              0
quarter            0
week               0
day_of_week        0
is_weekend         0
season             0
holiday            0
is_holiday         0
promotion_event    0
dtype: int64

In [161]:
calendar.head()

,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion
1,2024-01-02,2024,1,Q1,1,Tuesday,0,Winter,No Holiday,0,No Promotion
2,2024-01-03,2024,1,Q1,1,Wednesday,0,Winter,No Holiday,0,No Promotion
3,2024-01-04,2024,1,Q1,1,Thursday,0,Winter,No Holiday,0,No Promotion
4,2024-01-05,2024,1,Q1,1,Friday,0,Winter,No Holiday,0,No Promotion


## Inventory_snapshots

In [162]:
inventory.head()

,Snapshot_Date,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value
0,2024-01-01,SKU001,16,23,11,7,18,58420.96
1,2024-01-01,SKU002,29,12,4,5,10,163983.40
2,2024-01-01,SKU003,34,23,14,6,20,209060.22
3,2024-01-01,SKU004,34,4,7,5,12,42257.92
4,2024-01-01,SKU005,23,5,5,6,11,170945.89


In [163]:
inventory.shape

(4800, 8)

In [164]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Snapshot_Date    4800 non-null   object 
 1   SKU              4800 non-null   object 
 2   Current_Stock    4800 non-null   int64  
 3   On_Order         4800 non-null   int64  
 4   Lead_Time_Days   4800 non-null   int64  
 5   Safety_Stock     4800 non-null   int64  
 6   Reorder_Point    4800 non-null   int64  
 7   Inventory_Value  4800 non-null   float64
dtypes: float64(1), int64(5), object(2)
memory usage: 300.1+ KB


In [165]:
inventory.isnull().sum()

Snapshot_Date      0
SKU                0
Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64

In [166]:
inventory.duplicated().sum()

0

In [167]:
inventory.duplicated(subset = ["Snapshot_Date","SKU"]).sum()

0

In [168]:
inventory["Snapshot_Date"] = pd.to_datetime(inventory["Snapshot_Date"], errors = "coerce")

In [169]:
inventory.isnull().sum()

Snapshot_Date      0
SKU                0
Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64

In [170]:
numeric_cols = [
    "Current_Stock",
    "On_Order",
    "Lead_Time_Days",
    "Safety_Stock",
    "Reorder_Point",
    "Inventory_Value"
]

for col in numeric_cols:
    inventory[col] = pd.to_numeric(
        inventory[col],
        errors="coerce"
    )

In [171]:
inventory[inventory["Current_Stock"] < 0]
inventory[inventory["On_Order"] < 0]
inventory[inventory["Lead_Time_Days"] < 0]
inventory[inventory["Safety_Stock"] < 0]
inventory[inventory["Reorder_Point"] < 0]
inventory[inventory["Inventory_Value"] < 0]

,Snapshot_Date,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value


In [172]:
print("Shape:", inventory.shape)
print("Duplicates:", inventory.duplicated().sum())
print("SKU-Date duplicates:", inventory.duplicated(subset=["Snapshot_Date", "SKU"]).sum())
print(inventory.isnull().sum())
print(inventory.dtypes)

Shape: (4800, 8)
Duplicates: 0
SKU-Date duplicates: 0
Snapshot_Date      0
SKU                0
Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64
Snapshot_Date      datetime64[ns]
SKU                        object
Current_Stock               int64
On_Order                    int64
Lead_Time_Days              int64
Safety_Stock                int64
Reorder_Point               int64
Inventory_Value           float64
dtype: object


In [173]:
sales_skus = set(sales["SKU"].unique())
master_skus = set(sku["SKU"].unique())
print(sales_skus - master_skus)

set()


In [174]:
inventory_skus = set(inventory["SKU"].unique())
master_skus = set(sku["SKU"].unique())
inventory_skus - master_skus
set()

set()

In [175]:
valid_skus = set(sku["SKU"])
before = inventory["SKU"].nunique()
inventory = inventory[inventory["SKU"].isin(valid_skus)].copy()
print(f"Dropped {before - inventory['SKU'].nunique()} orphan SKUs -> {inventory.shape}")

Dropped 150 orphan SKUs -> (1200, 8)


In [177]:
sales.to_csv("../data/cleaned/sales_daily_clean.csv", index = False)
sku.to_csv("../data/cleaned/sku_master_clean.csv", index = False)
calendar.to_csv("../data/cleaned/calendar_clean.csv", index=False)
inventory.to_csv("../data/cleaned/inventory_snapshots_clean.csv", index=False)